# PI-KAN Hamiltonian Experiment + Extractor

This notebook now includes a dedicated extractor section to compare learned `Phi` with ground-truth Hamiltonian `H`.

Use the same `PIKANConfig.degree_n`, `seed`, and `hamiltonian_scale` as in `bautin_ideal.ipynb` (or any SymPy cross-check) so the checkpoint matches the unified Hamiltonian built there.


In [6]:
from pathlib import Path

import sympy as sp
import torch
import numpy as np
import matplotlib.pyplot as plt

from pi_kan.config import PIKANConfig
from pi_kan.train import train
from pi_kan.symbolic_extract import (
    restore_from_checkpoint,
    taylor_series_from_model,
    extract_symbolic_polynomial,
    hamiltonian_sympy,
    phi_h_comparison_metrics,
    save_symbolic_expression,
)


## 1) (Optional) Train

Run this only when you need a fresh checkpoint.


In [7]:
cfg = PIKANConfig()
force_retrain = False  # switch to True only when you really need a fresh run

ckpt_path = Path(cfg.output_dir) / 'best_model.pt'
cache_state_path = Path(cfg.output_dir) / 'phi_taylor_order8_state.json'

if ckpt_path.exists() and not force_retrain:
    print('Using existing checkpoint (no retraining):', ckpt_path)
else:
    print('Checkpoint missing or retrain requested -> starting training...')
    result = train(cfg)
    ckpt_path = Path(result['checkpoint_path'])
    print('Training complete. New checkpoint:', ckpt_path)

if cache_state_path.exists():
    print('Found Taylor cache state:', cache_state_path)
else:
    print('No Taylor cache state found yet.')


Using existing checkpoint (no retraining): /Users/user/uni/10-sem/OND/lab3/pi_kan/outputs/best_model.pt
Found Taylor cache state: /Users/user/uni/10-sem/OND/lab3/pi_kan/outputs/phi_taylor_order8_state.json


In [8]:
# Training is now controlled in the previous cell via `force_retrain`.
# Keep this cell as a placeholder.


## 2) Extractor (from existing checkpoint)

This is the main section for symbolic extraction and Phi-vs-H verification.


In [9]:
# Prefer GPU/ANE (MPS) for extractor, fallback to CPU if unavailable.
target_device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model, system = restore_from_checkpoint(
    str(ckpt_path),
    cfg,
    map_location=target_device,
    target_device=target_device,
)
print('Loaded checkpoint:', ckpt_path)
print('Extractor device:', target_device)


Loaded checkpoint: /Users/user/uni/10-sem/OND/lab3/pi_kan/outputs/best_model.pt
Extractor device: mps


In [10]:
# IMPORTANT:
# Direct extractor may contain tanh compositions for deep KAN;
# for a clean polynomial use Taylor extraction.
cache_tag = 'phi_taylor_order8'

try:
    phi_sym = taylor_series_from_model(
        model,
        cfg,
        order=8,
        threshold=cfg.symbolic_threshold,
        verbose=True,
        cache_dir=cfg.output_dir,
        cache_tag=cache_tag,
        resume=True,
    )
except Exception as exc:
    print(f'Taylor extraction failed on current device ({type(exc).__name__}); retrying on CPU...')
    model, system = restore_from_checkpoint(str(ckpt_path), cfg, map_location='cpu', target_device='cpu')
    phi_sym = taylor_series_from_model(
        model,
        cfg,
        order=8,
        threshold=cfg.symbolic_threshold,
        verbose=True,
        cache_dir=cfg.output_dir,
        cache_tag=cache_tag,
        resume=True,
    )

phi_sym = sp.simplify(sp.expand(phi_sym))

h_sym = sp.simplify(sp.expand(hamiltonian_sympy(system)))
delta_sym = sp.simplify(sp.expand(phi_sym - h_sym))

print('=== Phi_sym(x,y) ===')
print(phi_sym)
print('\n=== H(x,y) ===')
print(h_sym)
print('\n=== Delta = Phi_sym - H ===')
print(delta_sym)


[extractor] resumed from cache: done_terms=3/45
[extractor] Taylor order=8, total partials=45
[extractor] degree d=2 started
Taylor extraction failed on current device (RuntimeError); retrying on CPU...
[extractor] resumed from cache: done_terms=3/45
[extractor] Taylor order=8, total partials=45
[extractor] degree d=2 started


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
# Coefficient-wise absolute differences
x, y = sp.symbols('x y', real=True)
phi_poly = sp.Poly(sp.expand(phi_sym), x, y)
h_poly = sp.Poly(sp.expand(h_sym), x, y)
all_monoms = sorted(set(phi_poly.monoms()) | set(h_poly.monoms()), key=lambda t: (t[0]+t[1], t[0], t[1]))

rows = []
max_abs_diff = 0.0
sum_abs_diff = 0.0
for m in all_monoms:
    mon = x**m[0] * y**m[1]
    c_phi = float(sp.N(phi_poly.coeff_monomial(mon)))
    c_h = float(sp.N(h_poly.coeff_monomial(mon)))
    d = abs(c_phi - c_h)
    max_abs_diff = max(max_abs_diff, d)
    sum_abs_diff += d
    if d > 0:
        rows.append((m, c_phi, c_h, d))

print(f'monom_count_union={len(all_monoms)}')
print(f'monom_count_nonzero_diff={len(rows)}')
print(f'max_abs_coeff_diff={max_abs_diff:.12e}')
print(f'sum_abs_coeff_diff={sum_abs_diff:.12e}')

print('\nTop 20 coefficient diffs:')
for m, cp, ch, d in sorted(rows, key=lambda r: r[3], reverse=True)[:20]:
    print(f'  x^{m[0]} y^{m[1]}: phi={cp:.12e}, H={ch:.12e}, |diff|={d:.12e}')


In [ ]:
metrics = phi_h_comparison_metrics(model, system, radius=cfg.domain_radius, grid_n=cfg.eval_grid_n)
print('Phi vs H metrics on grid:')
for k, v in metrics.items():
    print(f'  {k}: {v:.6e}')


In [ ]:
txt_path, tex_path = save_symbolic_expression(phi_sym, cfg.output_dir)
print('Saved Phi symbolic files:')
print(' -', txt_path)
print(' -', tex_path)
